In [ ]:
import sys
# TO CHANGE
BASEDIR = "../../../../.."
sys.path.insert(0, BASEDIR)

In [ ]:
from pprint import pprint
import gc

In [ ]:
from src.kg_model.embeddings_model import EmbeddingsModelConfig, EmbeddingsModel
from src.db_drivers.vector_driver import EmbedderModel, EmbedderModelConfig
from src.utils import TripletCreator, NodeCreator, RelationCreator

## Пример работы с EmbeddingsModel-частью модели графа знаний

1. Инициализация класса, выполняющего функции embedder-модели

In [ ]:
EMBEDDER_MODEL_PATH = '../../../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL

In [ ]:
embedder_config = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
pprint(embedder_config)

In [ ]:
embedder = EmbedderModel(embedder_config)

In [ ]:
embedders_mapping = {
    'dense_nodes': embedder,
    'dense_triplets': embedder
}

In [ ]:
PASSAGE_EXAMPLE1 = "Шла саша по шоссе и сосала сушку."
QUERY_EXAMPLE1 = "Корабли лавировали да не вылавировали."

PASSAGE_EMBEDDING1 = embedder.encode_passages([PASSAGE_EXAMPLE1])
print("passage-emb size:", len(PASSAGE_EMBEDDING1[0]))
QUERY_EMBEDDING1 = embedder.encode_passages([QUERY_EXAMPLE1])
print("query-emb size:", len(QUERY_EMBEDDING1[0]))

2. Инициализация EmbeddingsModel-класса 

In [ ]:
graph_embeddings_config = EmbeddingsModelConfig()
pprint(graph_embeddings_config)

In [ ]:
graph_embeddings = EmbeddingsModel(embedders_mapping, graph_embeddings_config)

3. Добавление триплетов в векторную модель

In [ ]:
TRIPLET_EXAMPLE1 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='саша'), 
    relation=RelationCreator.create(r_type='simple', name='шла'), 
    end_node=NodeCreator.create(n_type='object', name='шоссе'))
TRIPLET_EXAMPLE2 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='саша'), 
    relation=RelationCreator.create(r_type='simple', name='сосала'), 
    end_node=NodeCreator.create(n_type='object', name='сушка'))
TRIPLET_EXAMPLE3 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='корабли'), 
    relation=RelationCreator.create(r_type='simple', name='выполняли'), 
    end_node=NodeCreator.create(n_type='object', name='лавировать'))
TRIPLET_EXAMPLE4 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='корабли'), 
    relation=RelationCreator.create(r_type='simple', name='не удалось'), 
    end_node=NodeCreator.create(n_type='object', name='лавировать'))

In [ ]:
pprint(TRIPLET_EXAMPLE1, width=200)
pprint(TRIPLET_EXAMPLE2, width=200)
pprint(TRIPLET_EXAMPLE3, width=200)
pprint(TRIPLET_EXAMPLE4, width=200)

In [ ]:
create_info = graph_embeddings.create_triplets([TRIPLET_EXAMPLE1, TRIPLET_EXAMPLE2, TRIPLET_EXAMPLE3, TRIPLET_EXAMPLE4])
print("Идентификаторы добавленных объектов в векторную модель:")
pprint(create_info)
print("Количество объектов в векторной моделе: ")
pprint(graph_embeddings.count_items(detailed=True))

In [ ]:
graph_embeddings.check_consistency()

4. Удаление триплетов из векторной модели

In [ ]:
graph_embeddings.delete_triplets([TRIPLET_EXAMPLE2, TRIPLET_EXAMPLE4])
print("Количество объектов в векторной моделе: ")
pprint(graph_embeddings.count_items(detailed=True))

In [ ]:
graph_embeddings.check_consistency()

5. Полное удаление содержимого векторной модели

In [ ]:
graph_embeddings.clear()
print("Количество объектов в векторной моделе: ")
pprint(graph_embeddings.count_items(detailed=True))

In [ ]:
graph_embeddings.close_connections()
del graph_embeddings
gc.collect()